In [15]:
# %%
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
import random

np.random.seed(42)
torch.manual_seed(42)

In [16]:
# %%
def parse_array(x):
    if isinstance(x, list):
        return np.array(x, dtype=np.float32)
    
    if isinstance(x, str):
        x = x.strip("[]")
        x = x.replace(",", " ")
        return np.fromstring(x, sep=" ", dtype=np.float32)
    
    return np.zeros(0, dtype=np.float32)

In [17]:
def build_trajectories(sensor_df):
    sensor_df = sensor_df.sort_values(["trackId", "timestamp"])
    
    trajs = {}
    for tid, g in sensor_df.groupby("trackId"):
        trajs[tid] = g[["latitude", "longitude"]].values.astype(np.float32)
    
    return trajs

In [18]:
# %%
def sample_subtracks(trajs, L=12, stride=3):
    subs = []
    
    for tid, traj in trajs.items():
        if len(traj) < L:
            continue
        
        for i in range(0, len(traj)-L+1, stride):
            subs.append({
                "track_id": tid,
                "seq": traj[i:i+L]
            })
    
    return subs

In [19]:
# %%
def expand_point(lat, lon, L=12, sigma=0.01):
    pts = []
    for _ in range(L):
        pts.append([
            lat + np.random.normal(0, sigma),
            lon + np.random.normal(0, sigma)
        ])
    return np.array(pts, dtype=np.float32)

In [20]:
# %%
def encode_report(row, L=12):
    lons = parse_array(row["coors.longitudes"])
    lats = parse_array(row["coors.latitudes"])
    
    n = min(len(lons), len(lats))
    
    # CASE 1: only 1 point → Gaussian expansion
    if n == 1:
        return expand_point(lats[0], lons[0], L)
    
    # CASE 2: normal trajectory
    if n >= 2:
        pts = np.stack([lats[:n], lons[:n]], axis=1)
        idx = np.linspace(0, n-1, L).astype(int)
        return pts[idx]
    
    return np.zeros((L,2), dtype=np.float32)

In [21]:
# %%
def preprocess_seq(seq):
    xy = seq[:, :2]
    
    xy_local = xy - xy[0]         # shape
    pos = xy.mean(axis=0)         # global position
    
    return xy_local, pos

In [22]:
# %%
class TrajectoryEncoder(nn.Module):
    def __init__(self, in_dim=2, d_model=64):
        super().__init__()
        
        self.input_proj = nn.Linear(in_dim, d_model)
        
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.pool = nn.AdaptiveAvgPool1d(1)
    
    def forward(self, x):
        x = self.input_proj(x)
        h = self.encoder(x)
        h = h.transpose(1,2)
        h = self.pool(h).squeeze(-1)
        return h


class DualEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.backbone = TrajectoryEncoder()
        
        self.shape_head = nn.Linear(64, 64)
        self.pos_head   = nn.Linear(64, 2)
    
    def forward(self, x):
        h = self.backbone(x)
        
        shape = F.normalize(self.shape_head(h), dim=1)
        pos   = self.pos_head(h)
        
        return shape, pos

In [23]:
def dtw_distance(a, b):
    n, m = len(a), len(b)
    dp = np.full((n+1, m+1), np.inf)
    dp[0,0] = 0
    
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = np.linalg.norm(a[i-1] - b[j-1])
            dp[i,j] = cost + min(dp[i-1,j], dp[i,j-1], dp[i-1,j-1])
    
    return dp[n,m]

In [24]:
# %%
def geo_dist(a, b):
    return np.linalg.norm(a - b)

In [25]:
# %%
def build_candidates(sub_tracks, report_seqs, radius=2.0, top_k=20):
    candidates = []
    
    report_pos = [r.mean(axis=0) for r in report_seqs]
    
    for s in sub_tracks:
        _, pos = preprocess_seq(s["seq"])
        
        dists = np.array([geo_dist(pos, rp) for rp in report_pos])
        
        idx = np.where(dists < radius)[0]
        if len(idx) == 0:
            idx = np.argsort(dists)[:top_k]
        
        candidates.append(idx)
    
    return candidates

In [26]:
# %%
def train_model(sub_tracks, report_seqs, candidates, epochs=5):
    
    net_t = DualEncoder()
    net_r = DualEncoder()
    
    opt = torch.optim.Adam(
        list(net_t.parameters()) + list(net_r.parameters()),
        lr=1e-3
    )
    
    for ep in range(epochs):
        total = 0
        
        for _ in range(200):
            batch_idx = np.random.choice(len(sub_tracks), 32)
            
            t_seq, r_pos_seq, r_neg_seq = [], [], []
            t_pos, r_pos_pos, r_neg_pos = [], [], []
            
            for i in batch_idx:
                seq = sub_tracks[i]["seq"]
                seq_p, pos = preprocess_seq(seq)
                
                cand = candidates[i]
                pos_id = cand[0]
                neg_id = random.randint(0, len(report_seqs)-1)
                
                r_seq = report_seqs[pos_id]
                r_seq_p, r_pos_mean = preprocess_seq(r_seq)
                
                r_neg = report_seqs[neg_id]
                r_neg_p, r_neg_mean = preprocess_seq(r_neg)
                
                t_seq.append(seq_p)
                r_pos_seq.append(r_seq_p)
                r_neg_seq.append(r_neg_p)
                
                t_pos.append(pos)
                r_pos_pos.append(r_pos_mean)
                r_neg_pos.append(r_neg_mean)
            
            t_seq = torch.tensor(t_seq, dtype=torch.float32)
            r_pos_seq = torch.tensor(r_pos_seq, dtype=torch.float32)
            r_neg_seq = torch.tensor(r_neg_seq, dtype=torch.float32)
            
            t_pos = torch.tensor(t_pos, dtype=torch.float32)
            r_pos_pos = torch.tensor(r_pos_pos, dtype=torch.float32)
            r_neg_pos = torch.tensor(r_neg_pos, dtype=torch.float32)
            
            e_t, p_t = net_t(t_seq)
            e_pos, p_pos = net_r(r_pos_seq)
            e_neg, p_neg = net_r(r_neg_seq)
            
            # ranking
            sim_pos = (e_t * e_pos).sum(1)
            sim_neg = (e_t * e_neg).sum(1)
            ranking = F.relu(1 - sim_pos + sim_neg).mean()
            
            # GEO LOSS (dominant)
            geo_loss = F.mse_loss(p_t, p_pos)
            geo_neg  = F.relu(2.0 - torch.norm(p_t - p_neg, dim=1)).mean()
            
            loss = ranking + 3.0 * geo_loss + 0.5 * geo_neg
            
            opt.zero_grad()
            loss.backward()
            opt.step()
            
            total += loss.item()
        
        print(f"Epoch {ep+1}: loss={total:.4f}")
    
    return net_t, net_r

In [27]:
# %%
def match_reports(sub_tracks, report_seqs, net_t, net_r, top_k=5):
    
    results = {}
    
    for rid, r in enumerate(report_seqs):
        r_seq, r_pos = preprocess_seq(r)
        
        r_seq = torch.tensor(r_seq).unsqueeze(0)
        e_r, p_r = net_r(r_seq)
        p_r = p_r.detach().numpy()[0]
        
        scores = []
        
        for s in sub_tracks:
            t_seq, t_pos = preprocess_seq(s["seq"])
            
            t_seq = torch.tensor(t_seq).unsqueeze(0)
            e_t, p_t = net_t(t_seq)
            p_t = p_t.detach().numpy()[0]
            
            geo = geo_dist(p_t, p_r)
            if geo > 3.0:
                continue
            
            sim = torch.dot(e_t.squeeze(), e_r.squeeze()).item()
            dtw = dtw_distance(s["seq"], r)
            
            score = 0.3*sim + 0.3*np.exp(-dtw) - 1.2*geo
            
            scores.append((s["track_id"], score))
        
        scores.sort(key=lambda x: -x[1])
        results[rid] = scores[:top_k]
    
    return results

In [28]:
# %%
def plot_matches(sensor_df, event_df, results, num_reports=5):
    
    fig = go.Figure()
    
    for rid in list(results.keys())[:num_reports]:
        row = event_df.iloc[rid]
        
        lons = parse_array(row["coors.longitudes"])
        lats = parse_array(row["coors.latitudes"])
        
        fig.add_trace(go.Scatter(
            x=lons, y=lats,
            mode="markers+lines",
            name=f"Report {rid}",
            line=dict(width=4)
        ))
        
        for rank, (tid, sc) in enumerate(results[rid]):
            track = sensor_df[sensor_df["trackId"] == tid]
            
            fig.add_trace(go.Scatter(
                x=track["longitude"],
                y=track["latitude"],
                mode="lines",
                name=f"T{tid} r{rank+1} ({sc:.2f})"
            ))
    
    fig.update_layout(
        title="Lat/Lon Matching (Geo-aware)",
        height=700
    )
    
    fig.show()

In [29]:
# %%
sensor_df = pd.read_parquet("sensor.parquet")
event_df  = pd.read_parquet("event.parquet")

trajs = build_trajectories(sensor_df)
sub_tracks = sample_subtracks(trajs)

report_seqs = [
    encode_report(r) for r in event_df.to_dict("records")
]

candidates = build_candidates(sub_tracks, report_seqs)

net_t, net_r = train_model(sub_tracks, report_seqs, candidates)

results = match_reports(sub_tracks, report_seqs, net_t, net_r)

plot_matches(sensor_df, event_df, results)

/tmp/ipykernel_13945/2680333825.py:43: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  t_seq = torch.tensor(t_seq, dtype=torch.float32)


Epoch 1: loss=457.3989
Epoch 2: loss=391.5582
Epoch 3: loss=391.9974
Epoch 4: loss=392.0850
Epoch 5: loss=391.9406
